# 03 — Burned area and burn severity
## The August 2024 Galičica wildfire

**Case study:** a major wildfire on Galičica began on **5 August 2024** and burned for roughly two weeks.  
In this practical we map the spectral change using Sentinel-2 and **dNBR**.

By the end you will have:
- comparable pre-fire and post-fire composites;
- pre/post NBR;
- dNBR;
- a simple burn-severity classification;
- area statistics;
- an optional comparison with an EFFIS reference layer.

> **Important:** dNBR classes are not universal truth. Thresholds should be validated for vegetation type, season, sensor and management purpose.

## 1. Imports and Earth Engine

In [ ]:
from pathlib import Path
import ee
import pandas as pd
import geopandas as gpd
import folium

GEE_PROJECT_ID = "ee-andreydara"

try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

print("Earth Engine ready.")

## 2. Study area and dates

The fire was reported from early to mid-August 2024. We therefore compare:
- a **pre-fire** summer composite;
- a **post-fire** composite after the main event.

The windows are deliberately broad enough to obtain useful cloud-free observations.

In [ ]:
AOI = ee.Geometry.Rectangle([20.78, 40.86, 21.12, 41.18])
CENTER = [41.02, 20.95]
ZOOM = 10

PRE_START  = "2024-06-01"
PRE_END    = "2024-08-04"

POST_START = "2024-08-19"
POST_END   = "2024-09-30"

MAX_CLOUD = 50

## 3. Prepare Sentinel-2 composites

In [ ]:
def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)
        .Or(scl.eq(8))
        .Or(scl.eq(9))
        .Or(scl.eq(10))
        .Or(scl.eq(11))
    )
    return img.updateMask(bad.Not())

def s2_composite(start_date, end_date):
    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
        .map(mask_s2_scl)
    )
    print(start_date, "to", end_date, "scenes:", col.size().getInfo())
    return col.median().clip(AOI)

pre  = s2_composite(PRE_START, PRE_END)
post = s2_composite(POST_START, POST_END)

## 4. Inspect pre-fire and post-fire imagery

In [ ]:
def add_ee_layer(m, ee_image, vis_params, name):
    map_id = ee_image.getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
    ).add_to(m)

m = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

vis_swir = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3500}

add_ee_layer(m, pre, vis_swir, "Pre-fire SWIR/NIR/Red")
add_ee_layer(m, post, vis_swir, "Post-fire SWIR/NIR/Red")

folium.LayerControl().add_to(m)
m

### Before calculating anything

Switch repeatedly between the pre- and post-fire layers.

- Where do you see the clearest change?
- Is every change necessarily fire?
- Which confounders could affect a two-date comparison?

## 5. Calculate NBR and dNBR

In [ ]:
pre_nbr  = pre.normalizedDifference(["B8", "B12"]).rename("NBR_pre")
post_nbr = post.normalizedDifference(["B8", "B12"]).rename("NBR_post")

# Conventional sign: positive values generally indicate a drop in NBR after fire.
dnbr = pre_nbr.subtract(post_nbr).rename("dNBR")

print("dNBR ready.")

In [ ]:
m2 = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

add_ee_layer(
    m2, dnbr,
    {
        "min": -0.25,
        "max": 0.8,
        "palette": ["2166ac", "f7f7f7", "fdae61", "d73027", "7f0000"],
    },
    "dNBR"
)

folium.LayerControl().add_to(m2)
m2

## 6. Mask obvious water and built-up areas

Our training AOI includes parts of the Ohrid/Prespa surroundings.  
For a cleaner wildfire exercise we use **ESA WorldCover 2021** to exclude:
- water (`80`);
- built-up (`50`).

This is only a convenience mask, not a fire-validation layer.

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
land_mask = worldcover.neq(80).And(worldcover.neq(50))

dnbr_land = dnbr.updateMask(land_mask)

## 7. Simple burn-severity classes

For teaching we start from commonly used dNBR ranges:

| dNBR | Training label |
|---:|---|
| < 0.10 | Unburned / very low change |
| 0.10–0.27 | Low |
| 0.27–0.44 | Moderate-low |
| 0.44–0.66 | Moderate-high |
| > 0.66 | High |

These thresholds are **starting points**, not universal ecological severity classes.

In [ ]:
severity = (
    ee.Image(0)
    .where(dnbr_land.gte(0.10).And(dnbr_land.lt(0.27)), 1)
    .where(dnbr_land.gte(0.27).And(dnbr_land.lt(0.44)), 2)
    .where(dnbr_land.gte(0.44).And(dnbr_land.lt(0.66)), 3)
    .where(dnbr_land.gte(0.66), 4)
    .updateMask(dnbr_land.mask())
    .rename("severity")
)

severity_names = {
    0: "Unburned / very low change",
    1: "Low",
    2: "Moderate-low",
    3: "Moderate-high",
    4: "High",
}

In [ ]:
m3 = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

add_ee_layer(
    m3, severity,
    {
        "min": 0,
        "max": 4,
        "palette": ["d9d9d9", "ffffb2", "fecc5c", "fd8d3c", "bd0026"],
    },
    "dNBR severity classes"
)

folium.LayerControl().add_to(m3)
m3

## 8. Calculate area by class

In [ ]:
area_image = ee.Image.pixelArea().divide(1e6).addBands(severity)

grouped = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
    geometry=AOI,
    scale=20,
    maxPixels=1e9,
    bestEffort=True,
).get("groups").getInfo()

area_df = pd.DataFrame(grouped).rename(columns={"sum": "area_km2"})
area_df["label"] = area_df["class"].map(severity_names)
area_df = area_df[["class", "label", "area_km2"]].sort_values("class")
area_df["area_km2"] = area_df["area_km2"].round(2)

area_df

### Don't over-interpret the table

A rectangle-based training AOI plus generic thresholds can classify non-fire changes as burned.  
The important questions are:

1. Does the mapped pattern agree with the known fire location?
2. Where does it disagree with reference data?
3. Why?

## 9. Optional comparison with the EFFIS reference layer

The trainers have an EFFIS burned-area layer for comparison.  
To keep the student repository light, we will use a **small clipped reference file** when it is ready.

Expected path:

`~/mystorage/fire-school/data/effis/galicica_effis.gpkg`

The cell below safely skips the comparison if that file is not present.

In [ ]:
EFFIS_PATH = Path.home() / "mystorage" / "fire-school" / "data" / "effis" / "galicica_effis.gpkg"

if EFFIS_PATH.exists():
    effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326")
    print("EFFIS features:", len(effis))
    display(effis.head())

    m4 = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")
    folium.GeoJson(
        effis,
        name="EFFIS reference polygons",
        style_function=lambda _: {
            "color": "black",
            "weight": 2,
            "fillOpacity": 0.05,
        },
    ).add_to(m4)

    add_ee_layer(
        m4, severity.updateMask(severity.gt(0)),
        {
            "min": 1,
            "max": 4,
            "palette": ["ffffb2", "fecc5c", "fd8d3c", "bd0026"],
        },
        "dNBR mapped change"
    )

    folium.LayerControl().add_to(m4)
    display(m4)
else:
    print("EFFIS clipped reference file not installed yet — skip this section.")

## 10. Interpretation exercise — 15 minutes

In pairs, answer:

1. Where is the strongest mapped change?
2. Does the severity pattern look spatially coherent?
3. Find one place that may be a **false positive**.
4. How could topography, phenology, cloud masking, compositing dates or land-cover type affect dNBR?
5. If EFFIS is available, identify one disagreement between EFFIS and our map.
6. Which product would you trust more, and **for what specific purpose**?

There is no requirement that the two maps match perfectly.

## 11. Stretch tasks

Choose one if you finish early:

### A — Change the post-fire period
Try a later post-fire window. How stable is the mapped severity?

### B — Change thresholds
Move one dNBR threshold by ±0.05 and recompute areas.

### C — Compare with NDVI change
Calculate:

`NDVI_pre - NDVI_post`

Does it delineate the burn scar as clearly as dNBR?

### D — Add a second reference
Compare with an independent burned-area or fire-history product.

## 12. Output for the Galičica capstone

Save or note:

- your final dNBR map;
- area by severity class;
- **three defensible findings**;
- **one important limitation**;
- **one management-relevant interpretation**.

This practical feeds directly into the Friday burned-area/severity group task.